# Fitting a cubic with PyTorch

Time to graduate from numpy + hand-coded gradient descent to **PyTorch** — the framework that nanoGPT itself is built on. PyTorch buys us three big things:

1. **Autograd** — we no longer need to derive $\partial \mathcal{L} / \partial \theta$ by hand. Just write the forward pass; PyTorch tracks every operation and computes gradients automatically.
2. **Optimizers** — well-tested implementations of SGD, Adam, etc., so we don't write the update rule ourselves.
3. **GPUs** — the same code can move to CUDA with a one-line change.

To keep things grounded, we'll fit a **cubic** model

$$f(x; a, b, c, d) = a x^3 + b x^2 + c x + d$$

to data drawn from a noisy cubic. Same idea as the linear/sine fits in `basics.ipynb`, just with more parameters and zero hand-derived calculus.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## The dataset

Generate a noisy cubic from known true coefficients $(a^*, b^*, c^*, d^*)$ so we can later check how close PyTorch gets to recovering them.

In [ ]:
rng = np.random.default_rng(seed=0)

# True cubic: y = 0.5 x^3 - 1.2 x^2 - 0.7 x + 1.0
a_true, b_true, c_true, d_true = 0.5, -1.2, -0.7, 1.0
sigma_noise = 1.5

n_samples = 60
x_np = np.linspace(-3.5, 3.5, n_samples)
y_clean = a_true * x_np**3 + b_true * x_np**2 + c_true * x_np + d_true
y_np = y_clean + rng.normal(0.0, sigma_noise, size=n_samples)

x_dense = np.linspace(x_np.min(), x_np.max(), 400)
y_dense_true = a_true * x_dense**3 + b_true * x_dense**2 + c_true * x_dense + d_true

# Vertical residual segments from each data point to the true cubic.
# Same trick as notebook 1: lay out triplets [(x_i, y_i), (x_i, y_clean_i), None]
# so a single Scatter trace draws disconnected line segments.
n = len(x_np)
rx = np.empty(3 * n, dtype=object)
ry = np.empty(3 * n, dtype=object)
rx[0::3] = x_np;   ry[0::3] = y_np
rx[1::3] = x_np;   ry[1::3] = y_clean
rx[2::3] = None;   ry[2::3] = None

# Noise floor: the MSE between data and the true cubic. No model can do better than this.
noise_floor_mse = float(np.mean((y_np - y_clean) ** 2))

fig = go.Figure()
fig.add_trace(go.Scatter(x=rx, y=ry, mode="lines",
                         name="residuals (data → true)",
                         line=dict(color="rgba(120,120,120,0.5)", width=1),
                         hoverinfo="skip"))
fig.add_trace(go.Scatter(x=x_np, y=y_np, mode="markers", name="data",
                         marker=dict(size=6, opacity=0.8)))
fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                         name="true cubic", line=dict(color="crimson", width=2)))
fig.update_layout(
    title=f"Cubic dataset — noise floor MSE (data ↔ true) = {noise_floor_mse:.3f}",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=750, height=420,
)
fig.show()

print(f"Noise-floor MSE (data vs. true cubic) = {noise_floor_mse:.3f}")
print("→ this is the lowest MSE any model could possibly reach on this data.")

## The model — a cubic with four parameters

Our model is a cubic polynomial:

$$f(x;\, a, b, c, d) \;=\; a\, x^3 \;+\; b\, x^2 \;+\; c\, x \;+\; d$$

That's **four parameters** to learn — one more than the quadratic we played with in notebook 1, where we still managed to draw the loss surface in 3D (two parameters on the floor, loss on the vertical axis).

Add the scalar loss $\mathcal{L}(a, b, c, d)$ to the four parameters and we now have a **5-dimensional optimization problem**. We can no longer visualize it — there's no 5D scatter plot. The geometry is still there: still bowls, ridges, saddles, the same gradient $\nabla \mathcal{L}$ pointing uphill, same gradient-descent recipe. We just can't draw the picture anymore.

This is the dimensionality story of *every* real neural network. GPT-2 (124M params) has a loss landscape in 124-million-plus-one dimensions. The shapes don't change; we just lose the ability to look at them, and have to trust the math instead.

## The PyTorch mental model

Three things to internalize, coming from numpy:

- A **tensor** (`torch.tensor`) is numpy's `ndarray` plus two extras: it lives on a device (CPU or GPU), and it can carry gradients.
- A **parameter** is just a tensor with `requires_grad=True`. PyTorch silently builds a graph of every operation involving it, and `loss.backward()` walks that graph to deposit gradients into each parameter's `.grad` attribute.
- The training loop becomes a four-step ritual: **forward → loss → `loss.backward()` → optimizer step → zero grads**.

We'll start with the most explicit, "no helpers" version below so you can see the mechanics. Then we'll redo it with `torch.nn.Module` + `torch.optim`, which is how it's usually written.

In [ ]:
# Convert data to tensors (float32, on CPU)
x_t = torch.tensor(x_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.float32)

# The four cubic coefficients, initialized to zero. requires_grad=True
# tells PyTorch to track operations on these tensors so it can compute
# d_loss / d_param later.
a = torch.zeros((), requires_grad=True)
b = torch.zeros((), requires_grad=True)
c = torch.zeros((), requires_grad=True)
d = torch.zeros((), requires_grad=True)

def forward(x):
    return a * x**3 + b * x**2 + c * x + d

def mse(y_pred, y):
    return ((y_pred - y) ** 2).mean()




In [ ]:
print(a,b,c,d)

In [ ]:
x_t

In [ ]:
forward(x_t)

In [ ]:
# Sanity check: one forward pass + one backward pass, just to see the gradients.
loss0 = mse(forward(x_t), y_t)
print(f"Initial loss: {loss0.item():.3f}")


In [ ]:
# Yes! Every tensor produced by an operation on requires_grad=True inputs
# carries a `.grad_fn` reference to the op that made it. Each grad_fn in turn
# has a `.next_functions` tuple pointing to its parents. Walk those and you
# get the *full* autograd graph — the thing that `loss.backward()` traverses
# in reverse.

def print_grad_graph(fn, depth=0, _seen=None, max_depth=20):
    if _seen is None:
        _seen = set()
    if fn is None or id(fn) in _seen or depth > max_depth:
        return
    _seen.add(id(fn))
    branch = "└─ " if depth else ""
    print("  " * depth + branch + type(fn).__name__)
    for next_fn, _ in getattr(fn, "next_functions", []):
        print_grad_graph(next_fn, depth + 1, _seen, max_depth)


print(f"loss0 = {loss0.item():.3f}   grad_fn = {loss0.grad_fn}\n")
print("Autograd graph (read top-down; leaves are the parameters):")
print_grad_graph(loss0.grad_fn)

print()
print("The `AccumulateGrad` leaves are a, b, c, d — that's where `loss.backward()`")
print("ultimately deposits gradients. Every node above them is an operation in our")
print("`forward` and `mse` functions.")
print()
print("For a *rendered* graph, install torchviz (and the graphviz system package):")
print("    uv add --extra notebooks torchviz   # then: sudo apt install graphviz")
print("    from torchviz import make_dot")
print("    make_dot(loss0, params={'a': a, 'b': b, 'c': c, 'd': d})")

In [ ]:
# Rendered version of the same autograd graph using torchviz.
#
# Requires the torchviz Python package AND the graphviz system binary:
#     uv sync --extra notebooks      # adds torchviz to the venv
#     sudo apt install graphviz      # provides the `dot` binary that does the actual rendering
#
# `make_dot` walks the same `.grad_fn` chain we printed above, and the optional
# `params` dict labels the parameter leaves so we see "a", "b", "c", "d" instead
# of raw memory addresses.

from torchviz import make_dot

graph = make_dot(loss0, params={"a": a, "b": b, "c": c, "d": d})
graph  # last expression -> Jupyter renders the SVG inline

In [ ]:
# Sanity check: one forward pass + one backward pass, just to see the gradients.
loss0.backward()
loss0 = mse(forward(x_t), y_t)
print(f"Initial loss: {loss0.item():.3f}")
print(f"grad a = {a.grad.item():.3f}   grad b = {b.grad.item():.3f}")
print(f"grad c = {c.grad.item():.3f}   grad d = {d.grad.item():.3f}")

In [ ]:
# Clear the .grad attributes so we start the real training loop with a clean slate.
for p in (a, b, c, d):
    p.grad = None

## The training loop (manual SGD)

For each step:

1. Compute the loss (forward pass).
2. Call `loss.backward()` — autograd populates `.grad` on every parameter.
3. Update each parameter: $\theta \leftarrow \theta - \eta \cdot \theta.\text{grad}$. We wrap this in `torch.no_grad()` so the update itself isn't tracked by autograd.
4. Zero the gradients (otherwise PyTorch *accumulates* them across steps — which is a feature, but not what we want here).

Note the learning rate `eta`: with this 4-parameter problem and inputs on a $\pm 3.5$ range, the cubic term's gradient is large, so we need a small `eta` to keep things stable. With `Adam` later we'll be able to crank it up.

In [ ]:
n_steps = 4000
eta = 1e-3

losses = []
for step in range(n_steps):
    # 1. forward
    y_pred = forward(x_t)
    loss = mse(y_pred, y_t)

    # 2. backward
    loss.backward()

    # 3. update (no_grad: don't track this in the autograd graph)
    with torch.no_grad():
        for p in (a, b, c, d):
            p -= eta * p.grad
            p.grad = None  # 4. zero grads for next step

    losses.append(loss.item())

    if step % 500 == 0 or step == n_steps - 1:
        print(f"step {step:5d}   loss = {loss.item():8.4f}")

print()
print(f"Recovered:  a={a.item():+.3f}, b={b.item():+.3f}, c={c.item():+.3f}, d={d.item():+.3f}")
print(f"True:       a={a_true:+.3f}, b={b_true:+.3f}, c={c_true:+.3f}, d={d_true:+.3f}")

## How did it do?

Plot the fitted cubic against the data, and the loss curve (on a log y-axis so you can see the early-step drop).

In [ ]:
with torch.no_grad():
    y_fit = forward(torch.tensor(x_dense, dtype=torch.float32)).numpy()

fit_fig = go.Figure()
fit_fig.add_trace(go.Scatter(x=x_np, y=y_np, mode="markers", name="data",
                             marker=dict(size=6, opacity=0.8)))
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                             name="true cubic",
                             line=dict(color="crimson", width=2, dash="dash")))
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_fit, mode="lines",
                             name="PyTorch fit",
                             line=dict(color="orange", width=3)))
fit_fig.update_layout(title="Cubic fit via PyTorch + autograd",
                      xaxis_title="x", yaxis_title="y",
                      template="plotly_white", width=800, height=440)
fit_fig.show()

loss_fig = go.Figure()
loss_fig.add_trace(go.Scatter(y=losses, mode="lines", name="loss",
                              line=dict(color="darkorange", width=2)))
loss_fig.update_layout(title="Training loss (manual SGD)",
                       xaxis_title="step", yaxis_title="MSE",
                       yaxis_type="log",
                       template="plotly_white", width=800, height=360)
loss_fig.show()

## The idiomatic version: `nn.Module` + `torch.optim`

Everything we just did by hand has a cleaner equivalent in PyTorch's `nn` and `optim` modules:

- `torch.nn.Module` is the base class for any model. It auto-registers `Parameter` attributes so `model.parameters()` returns them all — no more passing a tuple of four tensors around.
- `torch.optim.Adam` (or `SGD`, `RMSprop`, …) handles the update step *and* the zero-grad. Adam in particular adapts a per-parameter step size, which is why we can use a much larger `lr` than the manual loop tolerated.
- `torch.nn.MSELoss()` is the same `mean((pred - target)**2)` we wrote, but standardized.

Same problem, same data, just rewritten the way you'd see it in a real PyTorch codebase (including nanoGPT's `train.py`).

In [ ]:
class Cubic(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.a = torch.nn.Parameter(torch.zeros(()))
        self.b = torch.nn.Parameter(torch.zeros(()))
        self.c = torch.nn.Parameter(torch.zeros(()))
        self.d = torch.nn.Parameter(torch.zeros(()))

    def forward(self, x):
        return self.a * x**3 + self.b * x**2 + self.c * x + self.d


model = Cubic()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()

losses_v2 = []
for step in range(2000):
    y_pred = model(x_t)
    loss = loss_fn(y_pred, y_t)

    opt.zero_grad()
    loss.backward()
    opt.step()

    losses_v2.append(loss.item())

print(f"Adam recovery:  a={model.a.item():+.3f}, b={model.b.item():+.3f}, "
      f"c={model.c.item():+.3f}, d={model.d.item():+.3f}   (loss {losses_v2[-1]:.4f})")
print(f"True:           a={a_true:+.3f}, b={b_true:+.3f}, c={c_true:+.3f}, d={d_true:+.3f}")

with torch.no_grad():
    y_fit_v2 = model(torch.tensor(x_dense, dtype=torch.float32)).numpy()

compare_fig = go.Figure()
compare_fig.add_trace(go.Scatter(x=x_np, y=y_np, mode="markers", name="data",
                                 marker=dict(size=6, opacity=0.8)))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                                 name="true cubic",
                                 line=dict(color="crimson", width=2, dash="dash")))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_fit, mode="lines",
                                 name="manual SGD",
                                 line=dict(color="orange", width=2)))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_fit_v2, mode="lines",
                                 name="Adam",
                                 line=dict(color="seagreen", width=3)))
compare_fig.update_layout(title="Fits compared: manual SGD vs. nn.Module + Adam",
                          xaxis_title="x", yaxis_title="y",
                          template="plotly_white", width=800, height=440)
compare_fig.show()

loss_compare = go.Figure()
loss_compare.add_trace(go.Scatter(y=losses, mode="lines", name="manual SGD",
                                  line=dict(color="darkorange", width=2)))
loss_compare.add_trace(go.Scatter(y=losses_v2, mode="lines", name="Adam",
                                  line=dict(color="seagreen", width=2)))
loss_compare.update_layout(title="Training loss: manual SGD vs Adam",
                           xaxis_title="step", yaxis_title="MSE",
                           yaxis_type="log",
                           template="plotly_white", width=800, height=360)
loss_compare.show()